# ⏰ Notebook 5: Circadian Rhythm & Time-of-Day (MLP Neural Network)

## 1. Physiological Context & Plant Circadian Clock
Plants possess an internal biological clock (circadian rhythm) that cycles roughly every 24 hours.
Diurnal environmental changes (light, temperature, transpiration) modulate stomatal conductance, potassium ($K^+$) exchange, and membrane electrogenic proton pumps.
As a result, the plant's resting biopotential baseline exhibits rhythmic diurnal drift between morning, afternoon, and night.

### Objective:
- Train a **Multi-Layer Perceptron (MLP Neural Network)** to classify diurnal time-of-day phases:
  * **`morning`** (05:00 – 11:59)
  * **`afternoon`** (12:00 – 16:59)
  * **`night`** (21:00 – 04:59)


In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import classification_report, confusion_matrix

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.dpi'] = 120

df = pd.read_csv("plant_time_series_features.csv")
print("Time of Day Observations:")
print(df["time_of_day"].value_counts())


## 2. Diurnal Baseline Shift Across the 24-Hour Cycle
Let's visualize the baseline biopotential (`mean` and `median`) across morning, afternoon, and night recordings.


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

order = ["morning", "afternoon", "night"]
palette = {"morning": "#FFA726", "afternoon": "#29B6F6", "night": "#5C6BC0"}

sns.boxplot(data=df, x="time_of_day", y="mean", order=order, ax=ax1, palette=palette)
ax1.set_title("Mean Biopotential (ADC Count) Across Diurnal Phases", fontweight='bold')
ax1.set_ylabel("Mean ADC Count")
ax1.set_xlabel("Time of Day")

sns.boxplot(data=df, x="time_of_day", y="var", order=order, ax=ax2, palette=palette)
ax2.set_title("Biopotential Signal Variance Across Diurnal Phases", fontweight='bold')
ax2.set_ylabel("Signal Variance (ADC^2)")
ax2.set_xlabel("Time of Day")

plt.tight_layout()
plt.show()


## 3. Train Multi-Layer Perceptron (MLP Neural Network)
We train an MLP with two hidden layers: $(128, 64)$ with ReLU activations and adaptive Adam optimization.


In [ ]:
feature_cols = [
    "mean", "std", "var", "min", "max", "range", "median", 
    "rms", "mad", "p25", "p75", "iqr", "zcr", "skew", "kurt", 
    "peak_count", "low_freq_power", "mid_freq_power", "high_freq_power", 
    "spectral_entropy", "dominant_freq"
]

time_phases = ["morning", "afternoon", "night"]
time_to_idx = {t: i for i, t in enumerate(time_phases)}

X = df[feature_cols].values
y = df["time_of_day"].map(time_to_idx).values
groups = df["session_id"].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

sgkf = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=42)

mlp = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    alpha=0.001,
    batch_size=32,
    max_iter=300,
    random_state=42,
    early_stopping=True,
    validation_fraction=0.15
)

oof_preds_mlp = np.zeros(len(y), dtype=int)

for fold, (tr_idx, te_idx) in enumerate(sgkf.split(X_scaled, y, groups)):
    mlp.fit(X_scaled[tr_idx], y[tr_idx])
    oof_preds_mlp[te_idx] = mlp.predict(X_scaled[te_idx])

print("=== MLP Neural Network (Circadian Rhythms) Performance ===")
print(classification_report(y, oof_preds_mlp, target_names=time_phases))


## 4. Circadian Confusion Matrix Heatmap


In [ ]:
plt.figure(figsize=(7, 6))
cm_circ = confusion_matrix(y, oof_preds_mlp, normalize='true')
sns.heatmap(cm_circ, annot=True, fmt=".2%", cmap="Blues",
            xticklabels=time_phases, yticklabels=time_phases)
plt.title("MLP Neural Network: Circadian Phase Confusion Matrix", fontweight='bold')
plt.ylabel("True Phase")
plt.xlabel("Predicted Phase")
plt.tight_layout()
plt.show()


## 5. Neural Network Training Loss Curve


In [ ]:
plt.figure(figsize=(9, 4.5))
plt.plot(mlp.loss_curve_, color='#00897B', lw=2.5, label='Cross-Entropy Loss')
plt.title("MLP Neural Network Training Loss Convergence", fontweight='bold')
plt.xlabel("Training Iterations (Epochs)")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


## 6. First Hidden Layer Weight Activations
We inspect the learned input weights of the first layer to see which features most strongly drive neurons.


In [ ]:
plt.figure(figsize=(12, 6))
weights_first_layer = np.abs(mlp.coefs_[0]).mean(axis=1)
indices = np.argsort(weights_first_layer)[::-1]

plt.barh(range(len(feature_cols)), weights_first_layer[indices][::-1], color='#E64A19', alpha=0.85)
plt.yticks(range(len(feature_cols)), [feature_cols[i] for i in indices][::-1])
plt.title("MLP Input Layer Mean Absolute Weight Activations", fontweight='bold')
plt.xlabel("Mean Weight Magnitude")
plt.grid(True, linestyle=':')
plt.tight_layout()
plt.show()
